In [5]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import os
import json
import ast


# Downloading data
We are using a dataset based on `McAuley-Lab/Amazon-Reviews-2023`. It includes only products that were added on 2023 and also combines different categories in one dataset, which is exactly what we need. By default, it also includes embeddings for every product, while it would be really convenient to use them, it would restrict us to only using particular embedding model and may lead to potential anomalies, so we ignore them.

In [6]:
path = "../data/amazon_products.csv"

if not os.path.exists(path):
    dataset = load_dataset("milistu/AMAZON-Products-2023")
    df = dataset['train'].to_pandas()

    columns = ['parent_asin', 'date_first_available', 'title', 'description', 'main_category', 'average_rating', 'rating_number', 'price', 'features', 'details']

    df = df[columns]
    df.to_csv(path, columns=columns)

df = pd.read_csv(path, index_col=0, header=0)

df.head()

,parent_asin,date_first_available,title,description,main_category,average_rating,rating_number,price,features,details
0,B000044U2O,2023-04-29,Anomie & Bonhomie,Amazon.com\nFans of Scritti Politti's synth-po...,Digital Music,4.2,56.0,NaN,[],"{'Date First Available': 'April 29, 2023'}"
1,B0BT4CWWC9,2023-01-26,Sunshine On My Shoulders: The Best Of John Den...,"“Sunshine On My Shoulders” is a 2CD, 36-track ...",Digital Music,4.7,502.0,19.98,[],{'Package Dimensions': '5.55 x 4.92 x 0.51 inc...
2,B0BS4L5LP6,2023-01-11,18 Greatest Hits of 38 Special,Track Listings: 1. Rockin' Into The Night 2. ...,Digital Music,5.0,1.0,14.97,[],"{'Item Weight': '4 Ounces', 'Run time': '1 hou..."
3,B0BSPBBP89,2023-01-20,The Gift [CD],Second studio album by the multi-million-selli...,Digital Music,4.8,34.0,12.99,[],{'Package Dimensions': '5.59 x 4.8 x 0.47 inch...
4,B0BT1YG8MV,2023-01-20,ΤΗΕ ΒΟΟΤLΕG SΕRΙΕS VοΙ. ᛐ7 ᛐ996-ᛐ997 FRΑԌΜΕΝΤՏ...,"EU Edition 2CD,\nDISC ONE - TIME OUT OF MIND [...",Digital Music,3.6,5.0,43.99,[],"{'Manufacturer': 'Columbia Records, Sony Music..."


# Data cleaning and processing
We start by cleaning text data from unexpected characters

In [7]:
from data_processing.data_cleaning import clean_text, clean_corrupted


In [8]:
# clean columns with text data
columns_to_clean = ['title', 'description', 'features']
df[columns_to_clean] = df[columns_to_clean].map(clean_text)
df[columns_to_clean] = df[columns_to_clean].map(clean_corrupted)

# replace empty strings with NaN for easier processing
df = df.replace("", np.nan)
df = df.replace("[]", np.nan)

# convert features values from strings to lists
df['features'] = df['features'].str.strip("[]{}").str.split(", ")
df = df.dropna(thresh=6)

# convert details to dictionaries
df['details'] = df['details'].map(ast.literal_eval)

# convert date from strings to actual date format
df['date_first_available'] = pd.to_datetime(df['date_first_available'])


In [9]:
df.head()

,parent_asin,date_first_available,title,description,main_category,average_rating,rating_number,price,features,details
0,B000044U2O,2023-04-29,Anomie & Bonhomie,Amazon.com Fans of Scritti Politti's synth-pop...,Digital Music,4.2,56.0,NaN,NaN,"{'Date First Available': 'April 29, 2023'}"
1,B0BT4CWWC9,2023-01-26,Sunshine On My Shoulders: The Best Of John Den...,"Sunshine On My Shoulders is a 2CD, 36-track c...",Digital Music,4.7,502.0,19.98,NaN,{'Package Dimensions': '5.55 x 4.92 x 0.51 inc...
2,B0BS4L5LP6,2023-01-11,18 Greatest Hits of 38 Special,Track Listings: 1. Rockin' Into The Night 2. ...,Digital Music,5.0,1.0,14.97,NaN,"{'Item Weight': '4 Ounces', 'Run time': '1 hou..."
3,B0BSPBBP89,2023-01-20,The Gift [CD],Second studio album by the multi-million-selli...,Digital Music,4.8,34.0,12.99,NaN,{'Package Dimensions': '5.59 x 4.8 x 0.47 inch...
4,B0BT1YG8MV,2023-01-20,NaN,"EU Edition 2CD, DISC ONE - TIME OUT OF MIND [2...",Digital Music,3.6,5.0,43.99,NaN,"{'Manufacturer': 'Columbia Records, Sony Music..."


# Building a graph from Amazon products

In [10]:
df.dtypes

parent_asin                        str
date_first_available    datetime64[us]
title                              str
description                        str
main_category                      str
average_rating                 float64
rating_number                  float64
price                          float64
features                        object
details                         object
dtype: object

In [14]:
from rdflib import Graph, Literal, RDF, RDFS, URIRef, Namespace
from rdflib.namespace import SDO, XSD

In [15]:
# 1. Setup Graph and Namespaces
g = Graph()
products_ns = Namespace('http://products-kg.org/')
products_scheme = Namespace('http://products-kg.org/scheme/')

g.bind("schema", SDO)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)
g.bind("pkg", products_ns)
g.bind("pkgs", products_scheme)

# 2. Define the Ontology (Schema Level)
def define_ontology(graph):
    # Classes
    graph.add((products_scheme.AmazonProduct, RDF.type, RDFS.Class))
    graph.add((products_scheme.AmazonProduct, RDFS.subClassOf, SDO.Product))
    graph.add((products_scheme.AmazonProduct, RDFS.label, Literal("Amazon Marketplace Product")))

    # Custom Properties (where Schema.org isn't specific enough)
    graph.add((products_scheme.hasFeature, RDF.type, RDF.Property))
    graph.add((products_scheme.hasFeature, RDFS.label, Literal("Product Feature")))
    graph.add((products_scheme.hasFeature, RDFS.domain, products_scheme.AmazonProduct))
    graph.add((products_scheme.hasFeature, RDFS.range, XSD.string))

# 3. Data Mapping Function
def map_row_to_graph(graph, row):
    # Create a unique URI using the ASIN
    product_uri = products_ns[row['parent_asin']]

    # Basic Triplets
    graph.add((product_uri, RDF.type, products_scheme.AmazonProduct))
    graph.add((product_uri, SDO.identifier, Literal(row['parent_asin'])))
    graph.add((product_uri, SDO.name, Literal(row['title'])))
    graph.add((product_uri, SDO.description, Literal(row['description'])))
    graph.add((product_uri, SDO.releaseDate, Literal(row['date_first_available'].isoformat(), datatype=XSD.date)))
    graph.add((product_uri, SDO.category, Literal(row['main_category'])))

    # Pricing & Rating
    graph.add((product_uri, SDO.price, Literal(row['price'], datatype=XSD.float)))

    # Aggregate Rating (Blank Node or Nested Structure)
    rating_node = URIRef(product_uri + "/rating")
    graph.add((product_uri, SDO.aggregateRating, rating_node))
    graph.add((rating_node, RDF.type, SDO.AggregateRating))
    graph.add((rating_node, SDO.ratingValue, Literal(row['average_rating'], datatype=XSD.float)))
    graph.add((rating_node, SDO.ratingCount, Literal(row['rating_number'], datatype=XSD.integer)))

    # Handling 'features' (list of strings)
    if isinstance(row['features'], list):
        for feature in row['features']:
            graph.add((product_uri, products_scheme.hasFeature, Literal(feature)))

    # Handling 'details' (dictionary)
    if isinstance(row['details'], dict):
        for key, value in row['details'].items():
            # Creating a sanitized property name from the dictionary key
            prop_name = key.replace(" ", "")
            custom_prop = products_scheme[prop_name]
            graph.add((product_uri, custom_prop, Literal(value)))

# Initialize Ontology
define_ontology(g)

# Example usage with your data row:
# (Assuming df is your pandas DataFrame)
# for index, row in df.iterrows():
#     map_row_to_graph(g, row)

print(f"Ontology defined. Graph currently has {len(g)} triples.")
print(g.serialize(format="ttl"))


Ontology defined. Graph currently has 7 triples.
@prefix pkgs: <http://products-kg.org/scheme/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema: <https://schema.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

pkgs:AmazonProduct a rdfs:Class ;
    rdfs:label "Amazon Marketplace Product" ;
    rdfs:subClassOf schema:Product .

pkgs:hasFeature a rdf:Property ;
    rdfs:label "Product Feature" ;
    rdfs:domain pkgs:AmazonProduct ;
    rdfs:range xsd:string .


